### DLS background
The core of a DLS measurement is the analysis of the scattered light's intensity fluctuations. This is accomplished by constructing an intensity autocorrelation function, denoted as $g_2(\tau)$. This function measures the correlation between the intensity of scattered light at a time $t$, $I(t)$, and the intensity at a later time $t+\tau$, averaged over all time $t$. Mathematically, it is expressed as:

$$
g_2(\tau) = \frac{\langle I(t)I(t+\tau) \rangle}{\langle I(t) \rangle^2}
$$

where $\tau$ is the delay time. While the instrument directly measures the intensity autocorrelation, $g_2(\tau)$, the property that is directly related to the particle dynamics is the electric field autocorrelation function, $g_1(\tau)$:

$$
g_1(\tau) = \frac{\langle E(t)E(t+\tau) \rangle}{\langle I(t) \rangle}
$$

<!-- 
Connecting field and intensity correlations: the Siegert relation
and how to test it
http://www.kaiserlux.de/coldatoms/Articles/Siegert.pdf
 -->


Even though $g_1$ is not measured, it is related to $g_2$ through the Siegert equation:

$$
g_2(q, \tau) = 1 + \beta [g_1(q, \tau)]^2
$$

Where $\beta$ is an instrumental constant.  This relation allows us to extract the physically relevant $g_1(\tau)$ from the experimentally measured $g_2(\tau)$. The significance of $g_1(\tau)$ lies in its direct connection to the particle size distribution (PSD). For a perfectly monodisperse sample (where the PSD can be modeled as a Dirac delta function), the electric field autocorrelation function decays as a single exponential:

$$
g_1(\tau) = e^{-\Gamma \tau}
$$
 
where $\Gamma$ is the decay rate, which is itself related to the translational diffusion coefficient, $D$, of the particles by $\Gamma = D q^2$, where q is the magnitude of the scattering vector. The scattering vector depends on the laser wavelength, the refractive index of the solvent, and most crucially the scattering angle. For an arbitrary distribution of decay rates, $G{\left(\Gamma \right)}$, $g_1(\tau)$ is

$$
g_1(\tau) = \int\limits_{0}^{\infty} G{\left(\Gamma \right)} e^{- \Gamma \tau}\, d\Gamma
$$

Since there is also an angular dependence, and we can indeed make a measurement at multiple angles, we can modify the expression to account for the scattering vector by including it as an input variable and expressing the function as $g_1(q, \tau)$. As for determining the PSD, we can do so through the Stokes-Einstein equation that relates the particle radius with the diffusion coefficient which is itself related to the decay rate. And with this, we have made the connection between the measured $g_2(q, \tau)$ and the desired value - the PSD.

### Gaussian Mixture Model
In many real-world scenarios, a sample may not have a single, continuous distribution of sizes but may instead be composed of several distinct populations of particles (e.g., a mix of monomers and aggregates). In such cases, it is useful to model the overall decay rate distribution, $G(\Gamma)$, as a mixture model. This approach represents $G(\Gamma)$ as a weighted sum of the distributions for each individual component population:

$$
G(\Gamma) = \sum_{i=1}^k{\alpha_i G_i(\Gamma)}
$$

Here, k is the number of distinct populations in the mixture, $G_i(\Gamma)$ is the decay rate distribution corresponding to the i-th population, and $\alpha_i$ is the fractional intensity weighting of that population, with the condition that 
$$
\sum_{i=1}^k{\alpha_i} = 1, \qquad \alpha_i > 0 \quad \forall i
$$

We can substitute this mixture model back into the integral equation for the electric field autocorrelation function, $g_1(q, \tau)$. Due to the linear property of integration, the integral of a sum is equal to the sum of the integrals. This allows us to express the total autocorrelation function as a weighted sum of the autocorrelation functions from each population:

$$
g_1(q, \tau) = \int\limits_{0}^{\infty} \left(\sum_{i=1}^k{\alpha_i G_i(\Gamma)}\right) e^{- \Gamma \tau}\, d\Gamma = \sum_{i=1}^k{\alpha_i \left(\int\limits_{0}^{\infty} G_i(\Gamma) e^{- \Gamma \tau}\, d\Gamma\right)}
$$

Recognizing that the integral term is simply the autocorrelation function for the i-th component, which we can denote as $g_{1,i}(q, \tau)$, the equation simplifies to:

$$
g_1(q, \tau) = \sum_{i=1}^k{\alpha_i g_{1,i}(q, \tau)}
$$

We choose to model the components as normal distributions:
$$
G_i(\Gamma) = \frac{\sqrt{2} e^{- \frac{\left(\Gamma - q^{2} \mu_i \right)^{2}}{2 q^{4} \sigma_i^{2}}}}{2 \sqrt{\pi} q^{2} \sigma_i}
$$

Where $\mu_i, \sigma_i$ are the diffusion coefficient mean and standard deviation of the i-th component normal distribution. Fortuitously, this allows the integral to be expressed in a closed form:
$$
g_{1,i}(q, \tau) = \frac{ \left(\operatorname{erf}{\left(\frac{\sqrt{2} \left(\mu_i - q^{2} \sigma_i^{2} t\right)}{2 \sigma_i} \right)} + 1\right) e^{q^{2} t \left(- \mu_i + \frac{q^{2} \sigma_i^{2} t}{2}\right)}}{2}
$$

Strictly speaking, the normal distribution has a positive probability at every point in $(-\infty, \infty)$, which is non-physical as particle sizes cannot be negative, however by using a positive mean and ensuring that the standard deviation is small enough in comparison, the negative values are negligble, thus the coefficient of variation, $\frac{\sigma}{\mu}$ is enforced to be between $\frac{1}{2.5}$ and $\frac{1}{10}$, the lower bound being set so that distributions are not too narrow.

We use this Gaussian Mixture Model (GMM) to generate and fit to our synthetic data. while generating data is straightforward, a key challenge for fitting is to select the right number of components, $k$. Our method for determining this number is based on the HNMFk method, where we iterate through values for k, perform the fit, and run an analysis similar to an L-curve criterion to determine the best choice of value for $k$.

A normal distribution of diffusion coefficients corresponds to a PSD which is a one-sided reciprocal normal distribution. This is a strong and unrealistic assumption of a physical material, but does provide a valuable exercise as the fitting process still has the same complexity and ill-posed nature as would be seen when modelling a PSD with distributions such as log-normal, allowing to showcase the HNMFk method for determining the number of components, and it's closed form allows for more straightforward modelling as compared to a model that would require a numerical integration.

### Noise modelling
There are various sources and associated forms of noise that are present in DLS data, even in ideal experimental conditions. The most fundamental and inescapable of these is shot noise - an effect of the quantum nature of light, caused by photons arriving not as a constant stream. Instead the time it takes for a single photon to "fire" follows an exponential distribution, and thus the number of photons to arrive at the detector within a fixed window of time follows a Poisson distribution. The key factor here is the average rate of photons/sec from the laser, which for a HeNe laser is 45 kHz. The `add_poisson_noise` function shows how this noise is modelled.

In practice, the noise level can be controlled by repeated measurement and averaging of the data. This signal averaging allows us to scale thesignal-to-noise ratio by the square root of the number of measurements taken. While the noise level/SNR could be more directly controlled, doing so through the number of measurements allows for more direct interpretability

### multi-phase fitting
Directly fitting using parameters of normal curves (amplitudes, means, standard deviations) yielded subpar results, particularly for results with even minute noise. Fitting of a mixture of Dirac delta distributions, which have the parameters for amplitudes and "means" which were usually far better at finding those particular corresponding paramters of the normal distributions used to generate the data. It seems that the standard deviaton/widths of the normal curves are very sensitive to noise compared to the amplitudes and means. Since widths of the sub-populations in the PSD are of interest, there was motivation to at find a way to at estimate them as best as possible while not giving up the more promising amplitudes and means. 

For this reason, a multi-phase fitting process was developed:
1) the first phase fit is for a model of a delta distribution mixture
2) the second phase fit is for a model of a Gaussian mixture and it uses the amplitudes and means from the first phase solution as fixed fits only the standard deviations
3) The final phase fit is also a model of a Gaussian mixture except all the parameters are free and instead the solution from phase 2, with perturbations, are used as the initial points for the minimzation

This multi-phase approach showed a clear improvement over the straight shot fitting of normal curves and is what is the method presented here.

### selection of $k$
The model selection process is done by performing the fit a $N=100$ times for each value of $k$. Then, filtering is done to select the top 25% solutions by lowest loss value, and performing clustering on those components. For example if $k=3$ we will have $3 * 25$ total components, 3 for each solution, which are represented by a point in space for the parameters of each component, namely $(\alpha_i, \mu_i, \sigma_i)$ for the i-th component's amplitude/weight, mean, and standard deviation respectively. These component points are then clustered with a clustering algorithms like k-means where the number of clusters is $k=3$. For overparameterized models, the solution will lead to a minimal loss but there will be a wider variety of possible solutions that find that minimum, leading to poorer clustering. Underparameterized models will have tight clusters, but will not be able to reach as low of a loss. The optimal/correct value of $k$ will both find a minimal loss and have tight clusters.

### CONTIN/RILT fitting
The RILT program written in matlab as a rewrite of the CONTIN program written in Fortran (though not a true full translation as it does not include the automatic selection process for the hyperparameter $\alpha$) was written in python and adapted for multi-angle observations

### Solution evaluation
KL divergence

### To add
- optimizer details
- 


### 1. Fundamentals of Dynamic Light Scattering (DLS)

Dynamic Light Scattering (DLS) measures the size of particles in a suspension by analyzing the time-dependent fluctuations in scattered light intensity. These fluctuations arise from the Brownian motion of the particles. The analysis begins by constructing an intensity autocorrelation function, $g_2(\tau)$, which compares the scattered light intensity, $I(t)$, at a time $t$ with the intensity at a later time $t+\tau$:

$$
g_2(\tau) = \frac{\langle I(t)I(t+\tau) \rangle}{\langle I(t) \rangle^2}
$$
 
While $g_2(\tau)$ is what the instrument measures, the particle dynamics are described by the electric field autocorrelation function, $g_1(\tau)$. The two are related by the Siegert equation:

$$
g_2(q, \tau) = 1 + \beta [g_1(q, \tau)]^2
$$
 
where $\beta$ is an instrumental constant, and $q$ is the magnitude of the scattering vector, which depends on the laser wavelength, solvent refractive index, and scattering angle. This relationship allows us to extract the physically significant $g_1(\tau)$ from our experimental data.

The importance of $g_1(\tau)$ is its direct link to the distribution of particle diffusion coefficients. For a sample containing particles of a single size (a monodisperse system), $g_1(\tau)$ is a simple exponential decay:

$$
g_1(\tau) = e^{-\Gamma \tau}
$$
 
Here, $\Gamma$ is the decay rate, related to the translational diffusion coefficient $D$ by $\Gamma=Dq^2$. For a more realistic, polydisperse sample with a distribution of decay rates, $G(\Gamma)$, the function becomes an integral over all possible decay rates:

$$
g_1(\tau) = \int\limits_{0}^{\infty} G{\left(\Gamma \right)} e^{- \Gamma \tau}\, d\Gamma
$$

The ultimate goal is to determine the Particle Size Distribution (PSD). This is achieved by first solving the integral equation above to find the decay rate distribution, $G(\Gamma)$. From $G(\Gamma)$, we can determine the distribution of diffusion coefficients, which is then converted to a PSD using the Stokes-Einstein equation. The core challenge of DLS analysis, therefore, is reliably solving this ill-posed inverse problem to extract $G(\Gamma)$ from the measured $g_2(q, \tau)$.

### 2. The Gaussian Mixture Model (GMM)
Many samples consist of a mixture of distinct particle populations, such as monomers, dimers, and larger aggregates. To model such systems, we represent the overall decay rate distribution, $G(\Gamma)$, as a weighted sum of individual distributions:

$$
G(\Gamma) = \sum_{i=1}^k{\alpha_i G_i(\Gamma)}
\qquad \text{where} \qquad
\sum_{i=1}^k{\alpha_i} = 1, \qquad \alpha_i > 0 \quad \forall i
$$

Here, $k$ is the number of distinct populations, $G_i(\Gamma)$ is the decay rate distribution of the i-th population, and $\alpha_i$ is its fractional intensity contribution. Substituting this into the integral for $g_1(q, \tau)$ yields a sum of the autocorrelation functions from each population:

$$
g_1(q, \tau) = \sum_{i=1}^k{\alpha_i g_{1,i}(q, \tau)}
\qquad \text{where} \qquad
g_{1,i}(\tau) = \int\limits_{0}^{\infty} G_i{\left(\Gamma \right)} e^{- \Gamma \tau}\, d\Gamma
$$

For this work, we model each component distribution, $G_i(\Gamma)$, as a Normal distribution with a mean diffusion coefficient $\mu_i$ and standard deviation $\sigma_i$. While a Normal distribution is not strictly physical (as it allows for negative decay rates), it offers a significant practical advantage: the integral for $g_1(q, \tau)$ has a closed-form analytical solution:

$$
g_{1,i}(q, \tau) = \frac{ \left(\operatorname{erf}{\left(\frac{\sqrt{2} \left(\mu_i - q^{2} \sigma_i^{2} t\right)}{2 \sigma_i} \right)} + 1\right) e^{q^{2} t \left(- \mu_i + \frac{q^{2} \sigma_i^{2} t}{2}\right)}}{2}
$$

This avoids the need for numerical integration, simplifying the fitting process. To ensure physical relevance, we constrain the coefficient of variation ($\sigma_i/\mu_i$) to be between 0.1 and 0.4. This keeps the contribution from negative decay rates negligible while preventing the distributions from becoming unrealistically narrow. This GMM serves as the basis for generating and fitting our synthetic data, providing a robust framework to test our analysis methodology.

### 3. Noise Modeling
The primary source of noise in an ideal DLS experiment is shot noise, which stems from the discrete nature of photons arriving at the detector. The number of photons detected in a given time interval follows a Poisson distribution. We model this by adding Poisson-distributed noise to our synthetic correlograms, with the noise level determined by the average photon count rate (e.g., 45 kHz for a typical HeNe laser).

In a real experiment, the signal-to-noise ratio (SNR) is improved by averaging multiple measurements. We simulate this practice in our model, allowing us to control the noise level by adjusting the number of averaged runs. This provides a more experimentally relevant way to tune the SNR compared to simply scaling the noise amplitude directly.

### 4. A Multi-Phase Fitting Strategy
Directly fitting a GMM to DLS data is challenging because the parameters—particularly the distribution widths ($\sigma_i$)—are highly sensitive to noise. To overcome this instability, we developed a robust, multi-phase fitting procedure that progressively refines the parameter estimates.

- Phase 1: Peak Identification. We first fit the data with a simple mixture of Dirac delta functions. This model ignores the widths of the distributions and fits only for their positions ($\mu_i$) and amplitudes ($\alpha_i$). This simplified model is much less sensitive to noise and provides stable, reliable estimates for the primary peak locations and weights.

- Phase 2: Width Estimation. Next, we fit the data with the full GMM, but with a critical constraint: the amplitudes ($\alpha_i$) and means ($\mu_i$) are fixed to the values obtained in Phase 1. The fitting algorithm optimizes only for the standard deviations ($\sigma_i$), determining the width of each peak without the other parameters interfering.

- Phase 3: Final Refinement. Finally, we perform a full GMM fit where all parameters $(\alpha_i,\mu_i,\sigma_i)$ are allowed to vary. The crucial difference is that the initial guess for this final optimization is the complete solution obtained from Phase 2.

This structured approach, which uses the solution of a simpler model to intelligently guide the solution of a more complex one, dramatically improves the stability and accuracy of the final fit compared to a single-step optimization.

### 5. Robust Selection of the Number of Components ($k$)
A key challenge in mixture modeling is choosing the correct number of components, $k$. A model that is under-parameterized ($k$ is too small) will fit the data poorly. A model that is over-parameterized ($k$ is too large) may achieve a low loss value but often does so with non-physical, unstable solutions.

To find the optimal $k$, we employ a method inspired by the L-curve criterion that evaluates both the goodness-of-fit and the stability of the solution. For a range of potential $k$ values, we perform the full multi-phase fit numerous times (e.g., N=100) with different random initializations. We then analyze the results:

Filtering: For each k, we select the top 25 of solutions with the lowest final loss values.

Clustering: We treat the parameters of each component—$(\alpha_i,\mu_i,\sigma_i)$—as a point in 3D space. For a given k, this gives us $k \times (N \times 0.25)$ component points. We then run a clustering algorithm (e.g., k-means) on these points to find $k$ clusters.

Evaluation: The optimal $k$ is the one that exhibits both a low fitting error and tight, well-defined clusters. An over-parameterized model will find many different solutions that achieve a low error, resulting in diffuse, poorly formed clusters. The correct model will consistently converge to the same solution, resulting in tight clusters.

This method provides a robust criterion for model selection that avoids the common pitfall of overfitting.

### 6. Comparison with Regularized Inverse Laplace Transform (CONTIN/RILT)
[Note for further work]: The GMM approach should be validated against established methods. The next step is to implement a Python version of the classic CONTIN/RILT algorithm, adapted for multi-angle data. The same synthetic datasets should be analyzed with both the GMM method and the RILT implementation. A direct comparison of the recovered Particle Size Distributions, fitting residuals, and computational performance would provide a powerful benchmark and rigorously validate the performance of the proposed multi-phase GMM methodology.

## setup

| dia. nm                | 100          | 200         | 500         | 1000      | 0        |
|------------------------|--------------|-------------|-------------|-----------|----------|
|                        |              |             |             |           | 1mM NaCl |
| stock (solid fraction) | 1.998E-05    | 1.998E-05   | 1.998E-05   | 9.995E-06 | 0        |
| name                   | volumes (mL) |             |             |           |          |
| stock_100nm            | 1            | 0           | 0           | 0         | 0        |
| stock_200nm            | 0            | 1           | 0           | 0         | 0        |
| stock_500nm            | 0            | 0           | 1           | 0         | 0        |
| stock_1000nm           | 0            | 0           | 0           | 1         | 0        |
| mix_1                  | 0            | 0.8         | 0           | 0.2       | 0        |
| mix_2                  | 0            | 0.8         | 0           | 0.2       | 0.25     |
| mix_3                  | 0            | 0.8         | 0           | 0.2       | 1        |
| mix_4                  | 0            | 0.8         | 0           | 0.2       | 3        |
| mix_5                  | 0.333333333  | 0.333333333 | 0.333333333 | 0         | 0        |
| mix_6                  | 0.333333333  | 0.333333333 | 0.333333333 | 0         | 0.25     |
| mix_7                  | 0.333333333  | 0.333333333 | 0.333333333 | 0         | 1        |
| mix_8                  | 0.333333333  | 0.333333333 | 0.333333333 | 0         | 3        |
| mix_9                  | 0.333333333  | 0.333333333 | 0.333333333 | 0         | 9        |
| mix_10                 | 0.25         | 0.25        | 0.25        | 0.25      | 0        |
| mix_11                 | 0.25         | 0.25        | 0.25        | 0.25      | 0.25     |
| mix_12                 | 0.25         | 0.25        | 0.25        | 0.25      | 1        |
| mix_13                 | 0.25         | 0.25        | 0.25        | 0.25      | 3        |
| mix_14                 | 0.25         | 0.25        | 0.25        | 0.25      | 9        |
| mix_15                 | 0.25         | 0.25        | 0.25        | 0.25      | 19       |

In [ ]:
import base64
import io
import pickle
import functools
import itertools

import jax.numpy as jnp
import jax
jax.config.update("jax_platforms", "cpu")
import orthax

import matplotlib
import matplotlib.pyplot as plt
# import matplotlib.animation as animation
import pandas as pd
from scipy.io import loadmat, savemat
from scipy.stats import ranksums

# import plotly.graph_objects as go

from hnmf_tr_optimizer.hnmf_optimizer import HNMFOptimizer
from hnmf_tr_optimizer.clusts import result_analysis

from dls_model import InitParamsGenerator2, clustering_preprocess
# plt.style.use('Solarize_Light2')

# from IPython.display import Markdown, display



## models, optimizers, helpers, etc

In [ ]:
# Plotting theme setup


TEXT_COLOR = "white"
BG_COLOR = "black"

plt.rcParams["axes.facecolor"] = plt.rcParams["figure.facecolor"] = BG_COLOR
plt.rcParams["text.color"] = TEXT_COLOR
plt.rcParams["axes.labelcolor"] = TEXT_COLOR
plt.rcParams["xtick.color"] = TEXT_COLOR
plt.rcParams["ytick.color"] = TEXT_COLOR
plt.rcParams.update({
	"axes.grid" : True,
	"grid.color": "green",
	"grid.alpha": 0.35,
	"grid.linestyle": (0, (10, 10)),
})

# BETTER SIZES
DEFAULT_W, DEFAULT_H = (16, 9)
plt.rcParams["figure.figsize"] = [DEFAULT_W, DEFAULT_H]
plt.rcParams["font.size"] = 14
plt.rcParams["figure.dpi"] = 90

plt.style.use('dark_background')


In [ ]:
def scatter_vector(theta, lambda_0=633e-9, n=1.33, radians=False):
    """
    theta - scatter angle
    lambda_0 - lsder wavelength in meters
    n - refractive index of water
    """
    if not radians:
        theta = jnp.radians(theta)
    return (4 * jnp.pi * n/lambda_0) * jnp.sin(theta / 2)

def diffusion_coef(r, k_B=1.38e-23, T=298.15, eta=0.00089):
    """
    r - particle radius
    k_b - Boltzmann constant (J/K)
    T - Temperature (K)
    eta - Viscosity of water at room temerature (Pa*s)
    """
    return k_B * T / (6 * jnp.pi * eta * r)

def prep_data(datafile):
    df = pd.read_csv(datafile, delimiter='\t', header=None)
    df = df.iloc[1:] # remove strange first point
    d = jnp.array(df.to_numpy())

    t = d[:, 0]
    t *=  1e-3 # convert timestamps from ms to s

    # observation data is g2(t) - 1
    g2_minus1_obs = d[:, 1:].T

    div = jax.vmap(lambda x: x/x[0])
    g1_squared = div(g2_minus1_obs) # normalization by first term handles removing the beta term (roughly)

    # g1 = jnp.sqrt(jnp.maximum(g1_squared, 0)) # this line converts to g1
    g1 = jnp.where(
        jnp.greater_equal(g1_squared, 0),
        jnp.sqrt(g1_squared),
        -jnp.sqrt(-g1_squared)
    )

    theta = jnp.arange(30., 151, 5) # angles known in advance - in degrees
    q = scatter_vector(theta)
    return q, t, g1

def prep_data_g2(datafile):
    df = pd.read_csv(datafile, delimiter='\t', header=None)
    df = df.iloc[1:] # remove strange first point
    d = jnp.array(df.to_numpy())

    t = d[:, 0]
    t *=  1e-3 # convert timestamps from ms to s

    # observation data is g2(t) - 1
    g2_minus1_obs = d[:, 1:].T

    # div = jax.vmap(lambda x: x/x[0])
    # g1_squared = div(g2_minus1_obs) # normalization by first term handles removing the beta term (roughly)

    # # g1 = jnp.sqrt(jnp.maximum(g1_squared, 0)) # this line converts to g1
    # g1 = jnp.where(
    #     jnp.greater_equal(g1_squared, 0),
    #     jnp.sqrt(g1_squared),
    #     -jnp.sqrt(-g1_squared)
    # )

    theta = jnp.arange(30., 151, 5) # angles known in advance - in degrees
    q = scatter_vector(theta)
    return q, t, g2_minus1_obs

# for drawing normal curves
def normal_distribution_single(x, amplitude, mu, sigma):
    return amplitude * jnp.exp(-(x-mu)**2/(2*sigma**2))/jnp.sqrt(2*jnp.pi*sigma**2)

normal_distributions = jax.vmap(normal_distribution_single, in_axes=(None, 0, 0, 0))

def normal_distribution(possible_D, amp, mu, sig):
    whole = normal_distributions(possible_D, amp, mu, sig).sum(axis=0)
    return whole / jnp.sum(whole) # normalize for plotting


In [ ]:
### models

SCALING_CONST = 2.45e-7


######### Normal model ##########

def get_g1(t, nk, a, c):
    b = a**2/2
    d = jnp.sqrt(2)
    s_pi = jnp.sqrt(jnp.pi)
    x = (-a + c*t)/d
    y = jnp.where(
        jnp.greater_equal(x, 5),
        1/(x*s_pi),
        jax.scipy.special.erfc(x)*jnp.exp(x**2)
    )
    e = (nk/2)*jnp.exp(-b)
    return e*y

# vectorize along time dimension
all_g1 = jax.vmap(
    get_g1,
    in_axes=(0, None, None, None)
)

def source3_1(t, q, amp, mu, sig):
    ################
    const = SCALING_CONST
    # const = 1.0
    ################
    nk = amp
    sig = sig*const
    mu = mu*const
    a = mu/sig
    c = q**2*sig

    g = all_g1(t, nk, a, c)
    return g

by_Xs1 = jax.vmap(
    source3_1,
    in_axes=(None, None, 0, 0, 0)
)

source_matrix1 = jax.vmap(
    by_Xs1,
    in_axes=(None, 0, None, None, None)
)

def g1_matrix(q, t, amp, mu, sig):
    full = source_matrix1(t, q, amp, mu, sig)
    full = jnp.sum(full, axis=1)
    return full

def g2_minus1_matrix(q, t, amp, mu, sig, beta):
    g1 = g1_matrix(q, t, amp, mu, sig)
    g2_minus1 = beta * g1**2
    return g2_minus1

######### Dirac model ##########

def single_exp(q, t, D, amp):
    return amp * jnp.exp(-D * q**2 * t)

by_Xs = jax.vmap(
    single_exp,
    in_axes=(None, None, 0, 0)
)

by_t = jax.vmap(
    by_Xs,
    in_axes=(None, 0, None, None)
)

by_q = jax.vmap(
    by_t,
    in_axes=(0, None, None, None)
)

def g1_dirac(q, t, D, amp, const):
    D_ = D * const
    full = by_q(q, t, D_, amp)
    full = jnp.sum(full, axis=2)
    return full

def g2_minus1_matrix_dirac(q, t, D, amp, beta, const):
    amp = amp / jnp.sum(amp)
    g1 = g1_dirac(q, t, D, amp, const)
    g2_minus1 = beta * g1**2
    return g2_minus1


def gen_bounds_dirac(k):
    return (1e-9*jnp.ones(k),1e-9*jnp.ones(k),jnp.array([0.0])), (1e-3*jnp.ones(k), jnp.ones(k),jnp.array([1.0]))



In [ ]:
# quadrature based model

def g1_quadrature_(q, t, amp, mu, sig, scaling_const, laguerre_deg):
    x_lag, w_lag = orthax.laguerre.laggauss(laguerre_deg)
    f_lag = normal_distributions(scaling_const * x_lag, amp, mu*SCALING_CONST, sig*SCALING_CONST)
    return scaling_const * jnp.sum(w_lag * f_lag * jnp.exp((1-scaling_const*q**2*t) * x_lag))

g1_quadrature_by_t = jax.vmap(g1_quadrature_, in_axes = (0, None, None, None, None, None, None))
g1_quadrature_by_q = jax.vmap(g1_quadrature_by_t, in_axes = (None, 0, None, None, None, None, None))

def g2_minus1_quadrature(q, t, amp, mu, sig, beta, scaling_const, laguerre_deg):
    # amp = amp/jnp.sum(amp)
    return beta*jnp.square(g1_quadrature_by_q(q, t, amp, mu, sig, scaling_const, laguerre_deg).T)

# quad_obs = g2_minus1_quadrature(q, t, amp, mu, sig, 0.7, SCALING_CONST*1e-6, 150)
# old_obs = g2_minus1_matrix(q, t, amp, mu, sig, 0.7)


# quad_opt = HNMFOptimizer(
#     model_fn=g2_minus1_quadrature,
#     param_generator=InitParamsGenerator2(gen_bounds_normal_quadal),
#     bound_generator=gen_bounds_normal_quadal,
#     input_args = ('q', 't'),
#     param_args=('amp', 'mu', 'sig', 'beta'),
#     constants = {"scaling_const":SCALING_CONST*1e-6, "laguerre_deg": 25},
#     min_k=k,
#     max_k=k,
#     nsim=100
# )


In [ ]:
### Optimizers
min_k = 1
max_k = 3


# small_particle_bound = 1e-10 # 1 angstrom
# large_particle_bound = 1e-5 # 10 microns
def gen_bounds_std(num_sources):
    lower_bounds = (
        1e-9*jnp.ones(num_sources),
        jnp.array([1e-9])
    )
    upper_bounds = (
        1e-3*jnp.ones(num_sources),
        jnp.array([1.0])
    )
    return lower_bounds, upper_bounds

def gen_bounds_std_g1(num_sources):
    lower_bounds = (
        1e-9*jnp.ones(num_sources),
    )
    upper_bounds = (
        1e-3*jnp.ones(num_sources),
    )
    return lower_bounds, upper_bounds


def gen_bounds_normal_g1(num_sources):
    lower_bounds = (
        1e-9*jnp.ones(num_sources),
        1e-9*jnp.ones(num_sources),
        1e-9*jnp.ones(num_sources),
    )
    upper_bounds = (
        jnp.inf*jnp.ones(num_sources),
        1e-3*jnp.ones(num_sources),
        1e-3*jnp.ones(num_sources),
    )
    return lower_bounds, upper_bounds

# first phase optimizer - dirac model
# optimizer_dirac = HNMFOptimizer(
#     model_fn=g2_minus1_matrix_dirac,
#     param_generator=InitParamsGenerator2(gen_bounds_dirac),
#     bound_generator=gen_bounds_dirac,
#     input_args = ('q', 't'),
#     param_args=('D', 'amp', 'beta'),
#     constants = {"const": SCALING_CONST},
#     min_k=min_k,
#     max_k=max_k,
#     nsim=100
# )


# second phase optimizer(s) - normal model
# use values from first phase and only optimize to find a std of the gassians
# std_optimizers = {}
# for k in range(min_k, max_k + 1):
#     std_opt = HNMFOptimizer(
#         model_fn=g2_minus1_matrix,
#         param_generator=InitParamsGenerator2(gen_bounds_std),
#         bound_generator=gen_bounds_std,
#         input_args = ('q', 't', 'amp', 'mu'),
#         param_args=('sig', 'beta'),
#         constants = {},
#         min_k=k,
#         max_k=k,
#         nsim=100
#     )
#     std_optimizers[k] = (std_opt)

# std_g1_optimizers = {}
# for k in range(min_k, max_k + 1):
#     std_opt = HNMFOptimizer(
#         model_fn=g1_matrix,
#         param_generator=InitParamsGenerator2(gen_bounds_std_g1),
#         bound_generator=gen_bounds_std_g1,
#         input_args = ('q', 't', 'amp', 'mu'),
#         param_args=('sig',),
#         constants = {},
#         min_k=k,
#         max_k=k,
#         nsim=100
#     )
#     std_g1_optimizers[k] = (std_opt)


def extract_point(row):
    # extract source amplitudes and positions to use as points for clustering
    points = []
    sol = row['sol']
    D = sol[0]
    amp = sol[1]
    beta = sol[2]
    if isinstance(amp, float):
        D = jnp.array([D])
        amp = jnp.array([amp])
        beta = jnp.array([beta])
    else:
        D = jnp.array(D)
        amp = jnp.array(amp)
        beta = jnp.array(beta)
    amp = amp/jnp.sum(amp)
    for p in range(len(amp)):
        point = jnp.stack([D[p], amp[p], beta[p]]).tolist()
        points.append(point)
    return points

def clustering_preprocess(res):
    res = res.copy().groupby('num_sources', group_keys=False)[res.columns.tolist()].apply(filter_quantile)
    res['points'] = res.apply(extract_point, axis=1)
    return res

def filter_quantile(res, col_to_filter='fval', quantile=0.25):
    mod_col = res[col_to_filter].apply(lambda x: jnp.inf if jnp.isnan(x) else x)
    return res[
        mod_col < mod_col.quantile(q=quantile)
    ]

def process_res_dirac_(all_res, obs_size):
    Forclusts = clustering_preprocess(all_res)
    Forclusts = Forclusts.groupby('num_sources', group_keys=False)[Forclusts.columns.tolist()].apply(lambda group: result_analysis(
        group['points'].sum(),
        group['normF'].mean(),
        obs_size,
        group['num_sources'].iloc[0]
    ))
    Forclusts = Forclusts.set_index('num_source')

    return Forclusts



def extract_point_std(row):
    # extract source amplitudes and positions to use as points for clustering
    points = []
    sol = row['sol']
    sig = sol[0]
    beta = sol[1]
    beta = jnp.array(beta)
    if isinstance(sig, float):
        sig = jnp.array([sig])
    else:
        sig = jnp.array(sig)
    for p in range(len(sig)):
        point = sig.reshape(-1, 1)[p].tolist()
        points.append(point)
    return points

def clustering_preprocess_std(res):
    res = res.copy().groupby('num_sources', group_keys=False)[res.columns.tolist()].apply(filter_quantile)
    res['points'] = res.apply(extract_point_std, axis=1)
    return res

def process_res_std_(all_res, obs_size):
    Forclusts = clustering_preprocess_std(all_res)
    Forclusts = Forclusts.groupby('num_sources', group_keys=False)[Forclusts.columns.tolist()].apply(lambda group: result_analysis(
        group['points'].sum(),
        group['normF'].mean(),
        obs_size,
        group['num_sources'].iloc[0]
    ))
    Forclusts = Forclusts.set_index('num_source')

    return Forclusts

def extract_point_std_g1(row):
    # extract source amplitudes and positions to use as points for clustering
    points = []
    sol = row['sol']
    sig = sol[0]
    if isinstance(sig, float):
        sig = jnp.array([sig])
    else:
        sig = jnp.array(sig)
    for p in range(len(sig)):
        point = sig.reshape(-1, 1)[p].tolist()
        points.append(point)
    return points

def clustering_preprocess_std_g1(res):
    res = res.copy().groupby('num_sources', group_keys=False)[res.columns.tolist()].apply(filter_quantile)
    res['points'] = res.apply(extract_point_std_g1, axis=1)
    return res

def process_res_std_g1_(all_res, obs_size):
    Forclusts = clustering_preprocess_std_g1(all_res)
    Forclusts = Forclusts.groupby('num_sources', group_keys=False)[Forclusts.columns.tolist()].apply(lambda group: result_analysis(
        group['points'].sum(),
        group['normF'].mean(),
        obs_size,
        group['num_sources'].iloc[0]
    ))
    Forclusts = Forclusts.set_index('num_source')

    return Forclusts

from dls_model import clustering_preprocess as clustering_preprocess_std_normal

def process_res_normal_(all_res, obs_size):
    Forclusts = clustering_preprocess_std_normal(all_res)
    Forclusts = Forclusts.groupby('num_sources', group_keys=False)[Forclusts.columns.tolist()].apply(lambda group: result_analysis(
        group['points'].sum(),
        group['normF'].mean(),
        obs_size,
        group['num_sources'].iloc[0]
    ))
    Forclusts = Forclusts.set_index('num_source')

    return Forclusts


In [ ]:
def l_statistic(full_sols, clust_info, sill_threshold=0.6, p_threshold=0.05):
    p_values = {}
    errors = {}
    n_opt = 1
    for k in clust_info[clust_info['min_sillhouette_score'] > sill_threshold].index:
        current_errors = full_sols[full_sols['num_sources'] == k]['normF'].sort_values().to_list()
        errors[k] = current_errors

        # For the second valid k onwards, perform the statistical test
        if k > 1:
            # Get the errors from the previous valid k
            prev_errors = errors[k-1]
            
            # Wilcoxon rank-sum test to see if the new errors are significantly smaller
            # We use a one-sided test ('less') to check if the current error distribution
            # is stochastically less than the previous one.
            _, p_val = ranksums(current_errors, prev_errors, alternative='less')
            p_values[k] = p_val
            
            # If the result is significant, this k is a better model
            if p_val < p_threshold:
                n_opt = k
    return n_opt, p_values, errors

In [ ]:
def l_statistic2(full_sols, clust_info, observations, q, t, sill_threshold=0.6, p_threshold=0.05):
    p_values = {}
    errors = {}
    n_opt = 1
    for k in clust_info[clust_info['min_sillhouette_score'] > sill_threshold].index:
        best_amp, best_D, best_sig, best_beta = full_sols[full_sols['num_sources'] == k].sort_values('fval').iloc[0]['sol']
        recon = g2_minus1_matrix(q, t, best_amp, best_D, best_sig, best_beta)
        current_errors = jnp.zeros(len(q))
        # current_errors = []
        for qs in range(len(q)):
            s_segment = observations[qs, :]
            shat_segment = recon[qs, :]
            
            # Calculate the relative vector norm (error)
            error_norm = jnp.linalg.norm(shat_segment - s_segment)
            signal_norm = jnp.linalg.norm(s_segment)
            # Avoid division by zero if a signal segment is all zeros
            current_errors = current_errors.at[qs].set(error_norm / signal_norm if signal_norm > 0 else 0)

        errors[k] = current_errors


        # For the second valid k onwards, perform the statistical test
        if k > 1:
            # Get the errors from the previous valid k
            prev_errors = errors[k-1]
            
            # Wilcoxon rank-sum test to see if the new errors are significantly smaller
            # We use a one-sided test ('less') to check if the current error distribution
            # is stochastically less than the previous one.
            _, p_val = ranksums(current_errors, prev_errors, alternative='less')
            p_values[k] = p_val
            
            # If the result is significant, this k is a better model
            if p_val < p_threshold:
                n_opt = k
    return n_opt, p_values, errors

## run stuff

In [ ]:
# import some experimental data
q, t, observations_100 = prep_data("~/repos/DLS/Experimental_data_083122/stock_100nm.csv")
q, t, observations_200 = prep_data("~/repos/DLS/Experimental_data_083122/stock_200nm.csv")
q, t, observations_500 = prep_data("~/repos/DLS/Experimental_data_083122/stock_500nm.csv")
q, t, observations_1000 = prep_data("~/repos/DLS/Experimental_data_083122/stock_1000nm.csv")
q, t, mix_1 = prep_data("~/repos/DLS/Experimental_data_083122/mix_1.csv")
q, t, mix_2 = prep_data("~/repos/DLS/Experimental_data_083122/mix_2.csv")
q, t, mix_3 = prep_data("~/repos/DLS/Experimental_data_083122/mix_3.csv")
q, t, mix_4 = prep_data("~/repos/DLS/Experimental_data_083122/mix_4.csv")

q, t, observations_100_g2 = prep_data_g2("~/repos/DLS/Experimental_data_083122/stock_100nm.csv")
q, t, observations_200_g2 = prep_data_g2("~/repos/DLS/Experimental_data_083122/stock_200nm.csv")
q, t, observations_500_g2 = prep_data_g2("~/repos/DLS/Experimental_data_083122/stock_500nm.csv")
q, t, observations_1000_g2 = prep_data_g2("~/repos/DLS/Experimental_data_083122/stock_1000nm.csv")
q, t, mix_2_g2 = prep_data_g2("~/repos/DLS/Experimental_data_083122/mix_2.csv")
q, t, mix_1_g2 = prep_data_g2("~/repos/DLS/Experimental_data_083122/mix_1.csv")
q, t, mix_3_g2 = prep_data_g2("~/repos/DLS/Experimental_data_083122/mix_3.csv")
q, t, mix_4_g2 = prep_data_g2("~/repos/DLS/Experimental_data_083122/mix_4.csv")


mix_avg_g2 = jnp.mean(jnp.stack([mix_1_g2, mix_2_g2, mix_3_g2, mix_4_g2]), axis=0)


process_res_dirac = functools.partial(process_res_dirac_, obs_size=mix_1.size)
process_res_std = functools.partial(process_res_std_, obs_size=mix_1.size)
process_res_std_g1 = functools.partial(process_res_std_g1_, obs_size=mix_1.size)
process_res_normal = functools.partial(process_res_normal_, obs_size=mix_1.size)

In [ ]:
def generate_and_filter_distributions(amp_pairs, mean_params, std_params):
    # Generate combinations for the mean and standard deviation of both distributions
    mean_std_combinations = list(itertools.product(mean_params, std_params, mean_params, std_params))

    # Combine the user-provided amplitude pairs with the mean/std combinations
    all_combinations = []
    for amp1, amp2 in amp_pairs:
        for mean1, std1, mean2, std2 in mean_std_combinations:
            all_combinations.append([amp1, mean1, std1, amp2, mean2, std2])
    
    # Convert the list of combinations to a JAX array for efficient processing
    if not all_combinations:
        return jnp.array([]) # Return empty array if no combinations were generated
        
    all_combinations_jnp = jnp.array(all_combinations)
    _amp1, mean1, std1, _amp2, mean2, std2 = all_combinations_jnp.T
    a1 = (mean1 / 10) <= std1
    a2 = (mean1 / 2.5) >= std1
    a3 = (mean2 / 10) <= std2
    a4 = (mean2 / 2.5) >= std2
    a5 = (mean1 != mean2)

    valid_mask = jnp.all(
        jnp.array([
            a1, a2, a3, a4, a5
        ]),
        axis=0
    )

    # valid_mask = jnp.logical_and(jnp.logical_and(a1, a2), jnp.logical_and(a3, a4))

    return all_combinations_jnp[valid_mask]

radii = jnp.arange(50, 501, 50.) * 1e-9
_diff_coefs = diffusion_coef(radii)
scaled_diff_coefs = _diff_coefs / SCALING_CONST

amplitude_pairs = [[0.2, 0.8], [0.5, 0.5], [0.3, 0.7]]
mean_params = scaled_diff_coefs.tolist()
std_dev_params = jnp.linspace(5e-8, 1e-5, 6).tolist()

# Generate and filter the distributions
valid_params = generate_and_filter_distributions(
    amplitude_pairs, 
    mean_params, 
    std_dev_params
)

amp1, mean1, std1, amp2, mean2, std2 = valid_params.T
valid_params = jnp.stack([amp1, amp2, mean1, mean2, std1, std2]).T.reshape(valid_params.shape[0], 3, 2)



# radii = jnp.arange(50, 501, 50.) * 1e-9
# _diff_coefs = diffusion_coef(radii)
# scaled_diff_coefs = _diff_coefs / SCALING_CONST
# std = jnp.min(scaled_diff_coefs) * 0.8
# amplitude = 0.5

# i, j = jnp.triu_indices(len(scaled_diff_coefs), k=1)


# diff_pairs = jnp.stack([scaled_diff_coefs[i], scaled_diff_coefs[j]], axis=1)


# # first 25 have good separation examples
# diff_pairs = diff_pairs[:25]

# amps = jnp.repeat(amplitude, 2)
# stds = jnp.repeat(std, 2)


In [ ]:
# simulating shot (poisson) noise
def add_poisson_noise(g2_ideal, rand_key, average_counts_khz=1500, baseline=1.0):
    """
    g2_ideal - ideal g2 function
    average_counts_khz - average counts per channel in kHz
    seed - random seed for reproducibility
    """
    rand_key, subkey = jax.random.split(rand_key)
    mean_counts_per_channel = average_counts_khz * 100 # A proxy for total photon budget
    # The mean number of photons at each delay time τ is proportional to the ideal g2(τ)
    mean_photons_at_tau = mean_counts_per_channel * g2_ideal
    # Generate the noisy g2 data by drawing from a Poisson distribution
    # for each channel. This is the core of the shot noise simulation. 🎲
    noisy_counts = jax.random.poisson(subkey, mean_photons_at_tau)

    # Normalize the noisy counts to get the final noisy g2 function
    # The baseline of the noisy data is the average of the counts at long delay times
    noisy_baseline = jnp.mean(noisy_counts[:, -20:], axis=1) # Use last 20 channels for baseline
    g2_noisy = jax.vmap(lambda x, y: x/y)(noisy_counts, noisy_baseline)
    noisy_g2_minus_1 = g2_noisy - baseline  # Adjust the baseline to match the ideal g2

    return noisy_g2_minus_1, rand_key

def simulate_noisy_g2(
    g2_ideal: jnp.ndarray,
    rand_key: jax.random.PRNGKey,
    count_rate_khz: float,
    duration_s: float,
    baseline: float = 1.0,
    noise_scaling_factor: float = 0.1
) -> tuple[jnp.ndarray, jax.random.PRNGKey]:
    """
    Adds realistic Poisson noise to an ideal g2 autocorrelation function.

    This function simulates the shot noise inherent in a DLS experiment based on
    the instrument's count rate and the total measurement time.

    Args:
        g2_ideal: The ideal, noiseless g2 function (should not have the baseline subtracted).
                  Shape should be (batch, num_channels).
        rand_key: JAX random key for reproducibility.
        count_rate_khz: The average photon count rate in kHz (e.g., 20-45 from the manual).
        duration_s: The total duration of the experiment in seconds (e.g., 10, 30, 60).
        baseline: The theoretical baseline of the correlation function (typically 1.0).
        noise_scaling_factor: An empirical factor to match simulation to a real
                              correlator's output. It bridges the gap between total
                              photons and the statistical quality of the g2 function.
                              A value between 0.05 and 0.2 is a good starting point.

    Returns:
        A tuple containing:
        - noisy_g2_minus_1: The noisy g2 function with the baseline subtracted.
        - rand_key: The updated JAX random key.
    """
    # 1. Calculate the effective number of photon counts that contribute to the
    #    baseline of the correlation function. This is our "photon budget" and
    #    is the primary determinant of the noise level. It combines the
    #    instantaneous rate with the total measurement time.
    #    Total photons = count_rate_khz * 1000 * duration_s.
    #    The noise_scaling_factor adjusts this to better match the statistics
    #    of a real hardware correlator's averaging process.
    mean_counts_at_baseline = (
        count_rate_khz * 1000 * duration_s * noise_scaling_factor
    )

    # 2. The mean number of photons at each delay time τ is proportional to the ideal g2(τ).
    #    This creates the shape of the correlation function.
    mean_photons_at_tau = mean_counts_at_baseline * g2_ideal

    # 3. Generate the noisy data by drawing from a Poisson distribution for each channel.
    #    This is the core of the shot noise simulation.
    rand_key, subkey = jax.random.split(rand_key)
    noisy_counts = jax.random.poisson(subkey, mean_photons_at_tau)

    # 4. Normalize the noisy counts to get the final noisy g2 function.
    #    A robust method for finding the baseline of the noisy data is to average
    #    the counts from the last ~10% of the channels, where the function has decayed.
    num_channels_for_baseline = noisy_counts.shape[-1] // 10
    noisy_baseline = jnp.mean(noisy_counts[..., -num_channels_for_baseline:], axis=-1, keepdims=True)

    # Avoid division by zero if the baseline is somehow zero
    noisy_baseline = jnp.where(noisy_baseline == 0, 1.0, noisy_baseline)

    g2_noisy = noisy_counts / noisy_baseline

    # 5. Adjust by the theoretical baseline to center the result around 0.
    noisy_g2_minus_1 = g2_noisy - baseline

    return noisy_g2_minus_1, rand_key


def gen_data(
        q,
        t,
        valid_params,
        ensemble_size,
        gen_beta=0.7,
        baseline=1.0,
        rand_key=jax.random.key(1337),
        count_rate_khz=45.0,
        experiment_measurement_time_s=30.0,
        noise_scaling_factor=0.1
    ):
    clean_obs_list = []
    noisy_obs_list = []
    noise_errors = []
    snrs = []
    for i in range(valid_params.shape[0]):
        amp, mu, sig = valid_params[i]
        clean_g1 = g1_matrix(q, t, amp, mu, sig)
        g2_ideal = baseline + gen_beta*(clean_g1**2)


        # ensemble_observations = []
        # for _ in range(ensemble_size):
        #     noisy_g2_minus_1, rand_key = add_poisson_noise(g2_ideal, rand_key, average_counts_khz=average_counts_khz, baseline=baseline)
        #     ensemble_observations.append(noisy_g2_minus_1)
        # noisy_g2_minus_1 = jnp.mean(jnp.array(ensemble_observations), axis=0)

        # noisy_g2_minus_1, rand_key = add_poisson_noise(g2_ideal, rand_key, average_counts_khz=average_counts_khz, baseline=baseline)

        ensemble_observations = []
        for _ in range(ensemble_size):
            noisy_g2_minus_1, rand_key = simulate_noisy_g2(
                g2_ideal,
                rand_key,
                count_rate_khz=count_rate_khz,
                duration_s=experiment_measurement_time_s,
                baseline=baseline,
                noise_scaling_factor=noise_scaling_factor
            )
            ensemble_observations.append(noisy_g2_minus_1)
        noisy_g2_minus_1 = jnp.mean(jnp.array(ensemble_observations), axis=0)

        g2_ideal_minus1 = g2_ideal - baseline
        clean_obs_list.append(g2_ideal_minus1)

        snr = gen_beta / jnp.std((noisy_g2_minus_1 - g2_ideal_minus1))
        snrs.append(snr)

        r = g2_ideal_minus1 - noisy_g2_minus_1
        rmse = jnp.sqrt(jnp.sum(jnp.square(r)) / g2_ideal_minus1.size)
        noise_errors.append(rmse)

        noisy_obs_list.append(noisy_g2_minus_1)
        # noisy_obs_list.append(g2_ideal - baseline)

    return clean_obs_list, noisy_obs_list, noise_errors, snrs


# average_counts_khz = 1500
gen_beta = 0.7

ensemble_size = 30

clean_obs_list, noisy_obs_list, noise_errors, snrs = gen_data(
    q,
    t,
    valid_params,
    ensemble_size,
    gen_beta=gen_beta,
    baseline=1.0,
    rand_key=jax.random.key(1337),
    count_rate_khz=20.0, # average count rate for HeNe laser
    experiment_measurement_time_s=30.0, # total measurement time in seconds
    noise_scaling_factor=0.1
)

print(f"error due to noise (avg rmse): {jnp.average(jnp.array(noise_errors))}")
print(f"average SNR: {jnp.average(jnp.array(snrs))}")




book: noise and stochastic processes



criterion of fitting based on:
- difference of amplitudes
- distance between means
- widths
- noise level


In [ ]:
def g1_from_g2(g2_minus1_obs):
    """
    Convert g2(t) - 1 to g1(t) using the relation:
    g1(t) = sqrt(g2(t) - 1)
    This function assumes that g2_minus1_obs is normalized by the first term.
    """
    # Normalize by the first term
    div = jax.vmap(lambda x: x/x[0])
    g1_squared = div(g2_minus1_obs) # normalization by first term handles removing the beta term (roughly)

    # g1 = jnp.sqrt(jnp.maximum(g1_squared, 0)) # this line converts to g1
    return jnp.where(
        jnp.greater_equal(g1_squared, 0),
        jnp.sqrt(g1_squared),
        -jnp.sqrt(-g1_squared)
    )


# TODO: scale valid_params by 1000?
import numpy as np
from scipy.io import savemat
savemat('noisy_obs.mat', {'noisy_obs': [np.asarray(n) for n in noisy_obs_list], 't': np.asarray(t), 'q': np.asarray(q), 'valid_params': np.asarray(valid_params), 'noisy_obs_g1': [np.asarray(g1_from_g2(n)) for n in noisy_obs_list]})



In [ ]:
avg_snrs = []
avg_errors = []
ensemble_range = range(1, 50, 5)

for ensemble_size in ensemble_range:
    # print(f"Ensemble size: {ensemble_size}")
    noisy_obs_list, noise_errors, snrs = gen_data(q, t, valid_params, ensemble_size)

    avg_snrs.append(jnp.average(jnp.array(snrs)))
    avg_errors.append(jnp.average(jnp.array(noise_errors)))

plt.clf()
fig, ax1 = plt.subplots(figsize=(8, 5))

line1 = ax1.plot(ensemble_range, avg_errors, label='Average RMSE', color='tab:red')
ax2 = ax1.twinx()
line2 = ax2.plot(ensemble_range, avg_snrs, label='Average SNR', color='tab:blue')
line3 = ax2.plot(ensemble_range, jnp.square(jnp.array(avg_snrs))/1500, label='scaled sqrt Average SNR', color='tab:blue', linestyle='--')
ax1.set_xlabel('Ensemble Size')
ax1.set_ylabel('avg error')
ax2.set_ylabel('avg SNR')
ax1.set_title('Effect of Ensemble Size on Noise Error and SNR')
lines = line1 + line2 + line3
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels)
ax1.grid()
plt.show()


In [ ]:
# visualize a couple noisy observations

viz_num_pairs = 4
# viz_num_pairs = len(noisy_obs_list)

plt.clf()
fig, axs = plt.subplots(viz_num_pairs, 2, figsize=(12, 3*viz_num_pairs), dpi=150)

xx = jnp.linspace(0, 3e-5, 300)
for i in range(viz_num_pairs):
    amp, mu, sig = valid_params[i]
    srcs = normal_distribution(xx, amp, mu, sig)
    obs = noisy_obs_list[i]
    # draw each side by side
    # axs[i, 0].set_xscale('log')
    axs[i, 1].set_xscale('log')
    axs[i, 0].plot(xx, srcs)
    axs[i, 1].plot(t, obs.T)
    axs[i, 0].set_title(f"observations")
    axs[i, 1].set_title(f"srcs: {i+1} ({mu[0]:.2e}, {mu[1]:.2e})")
    axs[i, 0].set_xlabel("Diffusion coefficient (m^2/s)")
    axs[i, 1].set_xlabel("Time (s)")
    # axs[i, 0].set_ylabel("Probability density")
    axs[i, 1].set_ylabel("g2(t)")

fig.tight_layout()
plt.show()

In [ ]:
def rilt_observation_matrix(q, t, possible_D, x):
    A = jnp.exp(jnp.einsum('i,j,k->ijk', -possible_D, q**2, t))
    return jnp.einsum('i,ijk->jk', x, A)

def zero_at_ends_rilt(q, t, possible_D, x):
    x_ = x.at[0].set(0.0).at[-1].set(0.0)
    return rilt_observation_matrix(q, t, possible_D, x_)

def dummy_bounds(k):
    return (jnp.zeros(k),), (1e1 * jnp.ones(k),)


def L1_norm(x, alpha):
    return alpha * jnp.sum(jnp.abs(x))

L1_grad = jax.grad(L1_norm)
L1_hess = jax.hessian(L1_norm)

def L1_regularizer(x, alpha=1.0):
    return L1_norm(x, alpha), L1_grad(x, alpha), L1_hess(x, alpha)


def L2_norm(x, alpha):
    return alpha * jnp.sqrt(jnp.sum(jnp.square(x)))

L2_grad = jax.grad(L2_norm)
L2_hess = jax.hessian(L2_norm)

def L2_regularizer(x, alpha=1.0):
    return L2_norm(x, alpha), L2_grad(x, alpha), L2_hess(x, alpha)

# possible_D = jnp.logspace(-6.5, -4.5, 30) * SCALING_CONST
possible_D = jnp.linspace(1e-7, 3e-5, 40) * SCALING_CONST

contin_opt = HNMFOptimizer(
    model_fn=zero_at_ends_rilt,
    param_generator=InitParamsGenerator2(dummy_bounds),
    bound_generator=dummy_bounds,
    input_args = ('q', 't', 'possible_D'),
    param_args=('x'),
    constants = {},
    min_k=len(possible_D),
    max_k=len(possible_D),
    nsim=5,
    regularizer_fn=functools.partial(L2_regularizer, alpha=0.005)
    # regularizer_fn=functools.partial(L1_regularizer, alpha=1.0)
)



In [ ]:
rilt_sols_list = []


In [ ]:
for i in range(len(rilt_sols_list), len(noisy_obs_list)):
    all_res = contin_opt((q, t, possible_D), noisy_obs_list[i], opt_options={
        'fatol': 1e-14,
        'frtol': 0,
        'maxiter': 2000,
        'gatol': 1e-12
    })
    sols = jnp.stack(all_res.sort_values('fval')['sol'].apply(lambda l: l[0]).tolist())
    sols = sols.at[:, 0].set(0).at[:, -1].set(0)
    rilt_sols_list.append(sols)
    print(f"Pair {i+1} done\n\n")



In [ ]:
# set up the multi-phase optimization

optimizer_dirac_single = HNMFOptimizer(
    model_fn=g2_minus1_matrix_dirac,
    param_generator=InitParamsGenerator2(gen_bounds_dirac),
    bound_generator=gen_bounds_dirac,
    input_args = ('q', 't'),
    param_args=('D', 'amp', 'beta'),
    constants = {"const": SCALING_CONST},
    min_k=min_k,
    max_k=max_k,
    nsim=100
)


def gen_bounds_normal_std(num_sources):
    lower_bounds = (
        # 1e-9*jnp.ones(num_sources),
        # 1e-9*jnp.ones(num_sources),
        1e-9*jnp.ones(num_sources),
        # jnp.array([0.0])
    )
    upper_bounds = (
        # jnp.inf*jnp.ones(num_sources),
        # 1e-3*jnp.ones(num_sources),
        1e-3*jnp.ones(num_sources),
        # jnp.array([1.0])
    )
    return lower_bounds, upper_bounds




# std_opt = HNMFOptimizer(
#     # model_fn=g1_matrix,
#     model_fn=g2_minus1_matrix,
#     param_generator=InitParamsGenerator2(gen_bounds_normal_std),
#     bound_generator=gen_bounds_normal_std,
#     input_args = ('q', 't', 'amp', 'mu', 'beta'),
#     param_args=('sig',),
#     constants = {},
#     min_k=k,
#     max_k=k,
#     nsim=100
# )


def gen_bounds_normal_final(num_sources):
    lower_bounds = (
        1e-9*jnp.ones(num_sources),
        1e-9*jnp.ones(num_sources),
        1e-9*jnp.ones(num_sources),
        jnp.array([0.0])
    )
    upper_bounds = (
        jnp.inf*jnp.ones(num_sources),
        1e-3*jnp.ones(num_sources),
        1e-3*jnp.ones(num_sources),
        jnp.array([1.0])
    )
    return lower_bounds, upper_bounds


final_opt = HNMFOptimizer(
    # model_fn=g1_matrix,
    model_fn=g2_minus1_matrix,
    param_generator=InitParamsGenerator2(gen_bounds_normal_final),
    bound_generator=gen_bounds_normal_final,
    input_args = ('q', 't'),
    param_args=('amp', 'mu', 'sig', 'beta'),
    constants = {},
    min_k=min_k,
    max_k=max_k,
    nsim=100
)


inputs = (q, t)
opt_options = {
    'fatol': 1e-14,
    'frtol': 0,
    'maxiter': 2000,
    'gatol': 1e-12
}



In [ ]:
final_full_sols = []
final_clust_sols = []

# quad_final_full_sols = []
# quad_final_clust_sols = []

In [ ]:
class InitParamFeeder:
    def __init__(self, params_list_map):
        self.params_list_map = params_list_map
        self.counter = {k: 0 for k in params_list_map.keys()}

    def __call__(self, num_sources):
        params = self.params_list_map[num_sources][self.counter[num_sources]]
        self.counter[num_sources] += 1
        return params

rand_key = jax.random.key(1337)

for i in range(len(final_clust_sols), len(noisy_obs_list)):
    # observations = noisy_obs_list[i]
    observations = clean_obs_list[i]

    # phase 1 - fit dirac model, gives centers and amplitudes
    g2_res = optimizer_dirac_single((q, t), observations, opt_options=opt_options)
    clust_sol = process_res_dirac(g2_res)

    print("finished phase 1 for pair", i)


    # dirac_sols.append((D, amp, beta))

    # phase 2 - convert dirac to normal, only fit the standard deviation of the gaussians
    # centers and amplitudes are taken from the first phase and fixed

    params_to_feed = {}
    for k in clust_sol.index:
        D, amp, betas = clust_sol.loc[k]['centers']
        if not isinstance(D, jnp.ndarray):
            D = jnp.array(D, ndmin=1)
        if not isinstance(amp, jnp.ndarray):
            amp = jnp.array(amp, ndmin=1)
        if not isinstance(betas, jnp.ndarray):
            betas = jnp.array(betas, ndmin=1)
        beta = betas[0:1]
        std_opt = HNMFOptimizer(
            # model_fn=g1_matrix,
            model_fn=g2_minus1_matrix,
            param_generator=InitParamsGenerator2(gen_bounds_normal_std),
            bound_generator=gen_bounds_normal_std,
            input_args = ('q', 't', 'amp', 'mu', 'beta'),
            param_args=('sig',),
            constants = {},
            min_k=k,
            max_k=k,
            nsim=20
        )
        std_sols = std_opt((q, t, amp, D, beta), observations, opt_options=opt_options)
        sig_sol = process_res_std_g1(std_sols)['centers'].iloc[0][0]
        if not isinstance(sig_sol, jnp.ndarray):
            sig_sol = jnp.array(sig_sol, ndmin=1)

        nsim = 100
        params_to_feed[k] = []
        for j in range(nsim):
            rand_key, subkey1, subkey2, subkey3 = jax.random.split(rand_key, 4)
            D_ = D + jax.random.normal(subkey1, D.shape) * 0.05 * D
            sig_ = sig_sol + jax.random.normal(subkey1, sig_sol.shape) * 0.05 * sig_sol
            amp_ = amp + jax.random.normal(subkey2, amp.shape) * 0.05 * amp
            params_to_feed[k].append((amp_, D_, sig_, beta))

    print("finished phase 2 for pair", i)

    # final phase - fit with all parameters free


    final_opt = HNMFOptimizer(
        # model_fn=g1_matrix,
        model_fn=g2_minus1_matrix,
        param_generator=InitParamFeeder(params_to_feed),
        bound_generator=gen_bounds_normal_final,
        input_args = ('q', 't'),
        param_args=('amp', 'mu', 'sig', 'beta'),
        constants = {},
        min_k=min_k,
        max_k=max_k,
        nsim=nsim
    )

    final_g2_sols = final_opt((q, t), observations, opt_options=opt_options)
    final_full_sols.append(final_g2_sols)
    final_clust_sol = process_res_normal(final_g2_sols)
    final_clust_sols.append(final_clust_sol)

    with open('final_full_sols_noiseless.pkl', 'wb') as f:
        pickle.dump(final_full_sols, f)

    # quad_opt = HNMFOptimizer(
    #     model_fn=g2_minus1_quadrature,
    #     param_generator=InitParamFeeder(params_to_feed),
    #     bound_generator=gen_bounds_normal_final,
    #     input_args = ('q', 't'),
    #     param_args=('amp', 'mu', 'sig', 'beta'),
    #     constants = {"scaling_const":SCALING_CONST*1e-6, "laguerre_deg": 50},
    #     min_k=min_k,
    #     max_k=max_k,
    #     nsim=100
    # )
    # final_g2_sols = final_opt((q, t), observations, opt_options=opt_options)
    # quad_final_full_sols.append(final_g2_sols)
    # final_clust_sol = process_res_normal(final_g2_sols)
    # quad_final_clust_sols.append(final_clust_sol)

    # with open('quad_final_full_sols.pkl', 'wb') as f:
    #     pickle.dump(quad_final_full_sols, f)


    print(f"(noisy) Pair {i} done\n\n")


In [ ]:
### save solutions with pickle ###

# with open('final_full_sols.pkl', 'wb') as f:
#     pickle.dump(final_full_sols, f)


# save final_full_sols with pickle
# with open('quad_final_full_sols.pkl', 'wb') as f:
#     pickle.dump(quad_final_full_sols, f)


In [ ]:
### read in saved data ###

# with open('final_full_sols.pkl', 'rb') as f:
with open('final_full_sols_noiseless.pkl', 'rb') as f:
    final_full_sols = pickle.load(f)

final_clust_sols = [process_res_normal(final_g2_sols) for final_g2_sols in final_full_sols]


# with open('quad_final_full_sols.pkl', 'rb') as f:
#     quad_final_full_sols = pickle.load(f)

# quad_final_clust_sols = [process_res_normal(final_g2_sols) for final_g2_sols in quad_final_full_sols]


In [ ]:
# df = final_full_sols[0]
# jnp.stack(df[df['num_sources'] == 2]['sol'].apply(lambda x: final_opt.flatten(*x)[0]).tolist())


In [ ]:
amp, mu, sig = valid_params[0]
xx = jnp.linspace(0, 4e-5, 300)

normal_distributions(xx, amp, mu, sig).T.shape

In [ ]:
##################################
### run for ensemble_size = 5  ###
###       new noise method     ###
##################################

best_sols_matlab = loadmat("best_sols.mat")['best_sols']
best_sols_matlab = best_sols_matlab.reshape(best_sols_matlab.shape[0], 2, 3).swapaxes(1, 2)


xx = jnp.linspace(0, 4e-5, 300)

# num_rows = min(len(rilt_sols_list), len(final_sols))
num_rows = len(final_clust_sols)
plt.clf()
# fig, axs = plt.subplots(len(final_sols), 1, figsize=(16, 7*len(final_sols)), dpi=150)
fig, axs = plt.subplots(num_rows, 1, figsize=(16, 9*num_rows), dpi=150)

for i in range(num_rows):
    # observations = noisy_obs_list[i]
    observations = clean_obs_list[i]
    # amp_fin, D_fin, sig_fin, beta_fin = final_sols[i]
    ind = final_clust_sols[i]['aic_score'].argmin()
    n_srcs = final_clust_sols[i].iloc[ind].name
    amp_fin, D_fin, sig_fin = final_clust_sols[i].loc[n_srcs]['centers']
    if not isinstance(amp_fin, jnp.ndarray):
        amp_fin = jnp.array(amp_fin, ndmin=1)
    if not isinstance(D_fin, jnp.ndarray):
        D_fin = jnp.array(D_fin, ndmin=1)
    if not isinstance(sig_fin, jnp.ndarray):
        sig_fin = jnp.array(sig_fin, ndmin=1)

    beta_fin = 0.7

    # pred_srcs = normal_distribution(xx, amp, D, sig_sol)
    true_amp, true_mu, true_std = valid_params[i]
    true_srcs = normal_distributions(xx, true_amp, true_mu, true_std).T
    pred_srcs_final = normal_distributions(xx, amp_fin, D_fin, sig_fin).T

    clean_g2minus1 = g2_minus1_matrix(q, t, true_amp, true_mu, true_std, gen_beta)
    r = clean_g2minus1 - observations
    noise_level = 0.5 * jnp.sum(jnp.square(r))
    r = g2_minus1_matrix(q, t, amp_fin, D_fin, sig_fin, beta_fin) - observations
    fit_loss = 0.5*jnp.sum(jnp.square(r))

    # r = rilt_observation_matrix(q, t, possible_D, rilt_sols_list[i][0]) - observations
    # contin_loss = 0.5*jnp.sum(jnp.square(r))

    xx_ = xx * SCALING_CONST

    ax = axs[i]
    ax.plot(xx_, true_srcs, color='yellow', label='True distribution', linestyle=':')
    # ax.vlines(D, 0, amp*0.1, color='red', alpha=0.7, linestyle='-', label='phase 1 fit (dirac)')
    # ax.plot(xx, pred_srcs, color='teal', linestyle='-', label='phase 2 fit (width)')
    ax.plot(xx_, pred_srcs_final, color='green', linestyle='-', label='predicted distrubution (our model)')

    # if i < len(best_sols_matlab):
    #     amp_matlab, D_matlab, sig_matlab = best_sols_matlab[i]
    #     pred_srcs_matlab = normal_distribution(xx, amp_matlab, D_matlab, sig_matlab)
    #     ax.plot(xx_, pred_srcs_matlab, color='turquoise', linestyle='-.', label='matlab code prediction')


    kl_our_model = jnp.sum(jax.scipy.special.rel_entr(true_srcs, pred_srcs_final))

    # rilt_eval_points = possible_D/SCALING_CONST
    # for j in range(len(rilt_sols_list[i])):
    #     rilt_sol = rilt_sols_list[i][j]
    #     rilt_sols_xx = jnp.interp(xx, rilt_eval_points, rilt_sol/10)
    #     if j == 0:
    #         label = 'CONTIN solutions (scaled by .1)'
    #         true_srcs_eval = normal_distribution(rilt_eval_points, true_amp, true_mu, true_std)
    #         kl_contin = jnp.sum(jax.scipy.special.rel_entr(true_srcs_eval, (rilt_sol + jnp.finfo(float).eps))/jnp.sum(rilt_sol))
    #     else:
    #         label = None
    #     ax.plot(xx_, rilt_sols_xx, alpha = 0.5, color='turquoise', linestyle='-.', label=label)



        # ax.plot(possible_D / SCALING_CONST, rilt_sol, linestyle='--', label=f'CONTIN solution {j+1}')
    # rilt_sol = rilt_sols_list[i][0] # top result
    # rilt_sols_xx = jnp.interp(xx, possible_D/SCALING_CONST, rilt_sol/10)
    # ax.plot(xx, rilt_sols_xx, linestyle='-.', label=f'best CONTIN solution (scaled by .1)')
    # ax.plot(possible_D / SCALING_CONST, rilt_sol, linestyle='-.', label=f'best CONTIN solution')

    ax.set_xlabel('Diffusion Coefficient (m^2/s)')

    ax.legend()
    # ax.set_title(f"bimodal mix {i} - noise level: {noise_level:.4e}, loss (our model): {fit_loss:.4e}, loss (best CONTIN): {contin_loss:.4e}\nKL divergences - (our model): {kl_our_model:.2e}, (best CONTIN fit): {kl_contin:.2e}")

    # ax.set_title(f"bimodal mix {i} - noise level: {noise_level:.4e}, loss (our model): {fit_loss:.4e}, loss (best CONTIN): {contin_loss:.4e}\nKL divergences - (our model): {kl_our_model:.2e}")
    ax.set_title(f"bimodal mix {i} - noise level: {noise_level:.4e}, loss (our model): {fit_loss:.4e}\nKL divergences - (our model): {kl_our_model:.2e}, num srcs: {n_srcs}")

plt.show()


In [ ]:
import numpy as np
import miepython
import matplotlib.pyplot as plt

# --- 1. Define Input Parameters ---
m_particle = 1.59      # Refractive index of particle (e.g., Polystyrene)
n_medium = 1.33        # Refractive index of medium (e.g., Water)
wavelength_nm = 633    # Wavelength of the laser in nm
theta_deg = 173        # Scattering angle in degrees

# --- 2. Prepare Inputs for miepython ---
# The complex refractive index m = n_particle / n_medium
# Assuming non-absorbing particles, so the imaginary part is 0.
m = m_particle / n_medium

# Define a range of particle diameters to analyze
diameters_nm = np.linspace(1, 1000, 700)
radii_nm = diameters_nm / 2

# Calculate the dimensionless size parameter 'x'
x = 2 * np.pi * radii_nm * n_medium / wavelength_nm

# Calculate mu = cos(theta)
# The function requires the angle's cosine, not the angle itself.
mu = np.cos(np.deg2rad(theta_deg))

# --- 3. Calculate Mie Scattering Intensity ---
# This is the correct function call based on the docstring
intensity = []
for x_ in x:
    # Calculate the unpolarized intensity for each size parameter
    # The function returns the intensity normalized by the geometric cross-section
    intensity.append(miepython.i_unpolarized(m, x_, mu))
    miepython.in
# intensity = miepython.i_unpolarized(m, x, mu)

# --- 4. Plot the Results ---
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(diameters_nm, intensity, lw=2)
ax.set_xlabel("Particle Diameter (nm)", fontsize=12)
ax.set_ylabel("Normalized Unpolarized Intensity (a.u.)", fontsize=12)
ax.set_title(f"Mie Scattering Intensity at {theta_deg}° for Polystyrene in Water", fontsize=14)
# ax.set_yscale('log')
ax.grid(True, which="both", ls="--")

plt.show()

In [ ]:
x

In [ ]:
import numpy as np
import miepython
import matplotlib.pyplot as plt

# --- Parameters (same as before) ---
m = 1.59 / 1.33
wavelength_nm = 633
theta_deg = 173
mu = np.cos(np.deg2rad(theta_deg))

# --- Data Generation (same as before) ---
diameters_nm = np.linspace(10, 1000, 500)
radii_nm = diameters_nm / 2
x = 2 * np.pi * radii_nm * 1.33 / wavelength_nm
# intensity = miepython.i_unpolarized(m, x, mu)
intensity = []
for x_ in x:
    # Calculate the unpolarized intensity for each size parameter
    # The function returns the intensity normalized by the geometric cross-section
    intensity.append(miepython.i_unpolarized(m, x_, mu))


# --- Create new plots to inspect the Rayleigh region ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Inspecting the 'Flat' Region (d < 100 nm)", fontsize=16)

# Plot 1: The original Log-Lin view (zoomed in)
ax1.plot(diameters_nm, intensity)
# ax1.set_yscale('log')
ax1.set_xlim(0, 100)
ax1.set_ylim(1e-5, 0.2) # Adjust ylim to see the curve
ax1.set_title("Original Log-Lin View")
ax1.set_xlabel("Particle Diameter (nm)")
ax1.set_ylabel("Normalized Intensity")
ax1.grid(True, which="both", ls="--")

# Plot 2: A Log-Log view to reveal the power law
ax2.plot(diameters_nm, intensity)
ax2.set_xscale('log')
# ax2.set_yscale('log')
ax2.set_xlim(10, 100)
ax2.set_ylim(1e-5, 0.2)
ax2.set_title("Log-Log View Reveals the Power Law")
ax2.set_xlabel("Particle Diameter (nm)")
ax2.set_ylabel("Normalized Intensity")
ax2.grid(True, which="both", ls="--")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
def _single_bspline_eval(x, control_points, knots, degree):
    """Helper function to evaluate a B-spline for a single scalar x."""
    n = control_points.shape[0]
    
    # Initialize basis functions for degree 0
    # N_{i,0}(x) = 1 if knots[i] <= x < knots[i+1], else 0
    basis_funcs = (x >= knots[:n]) & (x < knots[1:n+1])
    
    # Recursively compute basis functions for higher degrees
    for d in range(1, degree + 1):
        # Temporarily store the basis functions from the previous degree
        prev_basis = basis_funcs
        # We need n basis functions of degree d, so we size the array for it
        basis_funcs = jnp.zeros(n)
        
        # --- First Term of the recursion ---
        # Denominator: knots[i+d] - knots[i]
        denom1 = knots[d:d+n] - knots[:n]
        # Handle division by zero for coincident knots
        term1 = jnp.where(
            denom1 > 0,
            (x - knots[:n]) / denom1 * prev_basis,
            0.0
        )
        
        # --- Second Term of the recursion ---
        # Denominator: knots[i+d+1] - knots[i+1]
        denom2 = knots[d+1:d+n+1] - knots[1:n+1]
        # Handle division by zero for coincident knots
        term2 = jnp.where(
            denom2 > 0,
            (knots[d+1:d+n+1] - x) / denom2 * prev_basis,
            0.0
        )
        
        # The new basis functions are the sum of two terms from the previous degree
        # N_{i,d} depends on N_{i,d-1} and N_{i-1,d-1}
        # To align terms correctly for summation:
        # basis_funcs[i] = term1[i] (from N_{i,d-1}) + term2[i-1] (from N_{i-1,d-1})
        basis_funcs = basis_funcs.at[1:].add(term2[:-1])
        basis_funcs = basis_funcs.at[:].add(term1)

    # Final spline value is the dot product of control points and basis functions
    res = jnp.dot(control_points, basis_funcs)
    return jnp.reshape(res, (1, -1))

# bspline = jax.jit(jax.vmap(_single_bspline_eval, in_axes=(0, None, None, None)), static_argnums=(3,))

@functools.partial(jax.jit, static_argnums=(3,))
def bspline(x, control_points, knots, degree):
    return jax.vmap(_single_bspline_eval, in_axes=(0, None, None, None))(x, control_points, knots, degree)



In [ ]:
from scipy.stats import norm

domain_start, domain_end = -4.0, 4.0
x_data = jnp.linspace(domain_start, domain_end, 200)
y_data = jnp.array(norm.pdf(x_data, loc=0, scale=1))

degree = 5
num_control_points = 10

num_total_knots = num_control_points + degree + 1
num_interior_knots = num_total_knots - 2 * degree

knots = jnp.linspace(domain_start, domain_end, num_total_knots)

# def gen_bounds_spline_dummy(k):
#     """Generates dummy bounds for the spline parameters."""
#     # Controls can be any real number, knots are clamped to the domain
#     lower_bounds = (jnp.zeros(num_control_points), domain_start*jnp.ones(num_total_knots))
#     upper_bounds = (jnp.inf*jnp.ones(num_control_points), domain_end*jnp.ones(num_total_knots))
#     return lower_bounds, upper_bounds

def gen_bounds_spline_dummy(k):
    """Generates dummy bounds for the spline parameters."""
    # Controls can be any real number, knots are clamped to the domain
    lower_bounds = (-100.0*jnp.ones(num_control_points), )
    upper_bounds = (jnp.inf*jnp.ones(num_control_points), )
    return lower_bounds, upper_bounds

spline_opt = HNMFOptimizer(
    model_fn=bspline,
    param_generator=InitParamsGenerator2(gen_bounds_spline_dummy),
    bound_generator=gen_bounds_spline_dummy,
    input_args=('x'),
    param_args=('control_points',),
    constants = {'degree': degree, 'knots': knots},
    min_k=1,
    max_k=1,
    nsim=20
)

ress = spline_opt(x_data, y_data, opt_options={
    'fatol': 1e-14,
    'frtol': 0,
    'maxiter': 2000,
    'gatol': 1e-12
})

ress = ress.sort_values('fval')


# final_control, final_knots = ress.iloc[0]['sol']
final_control = ress.iloc[0]['sol'][0]
final_knots = knots  # Use the fixed knots from the optimization
y_final = bspline(x_data, final_control, final_knots, degree)


plt.figure(figsize=(12, 8))
# Plot curves
plt.plot(x_data, jnp.squeeze(y_data), 'r--', label='Target Normal Curve', lw=2)
plt.plot(x_data, jnp.squeeze(y_final), 'b-', label='Fitted B-Spline (Trained)', lw=2.5)

# Plot initial and final knot positions
# plt.scatter(initial_knots, jnp.full_like(initial_knots, -0.02), marker='|', color='gray', s=100, label='Initial Knots')
plt.scatter(final_knots, jnp.full_like(final_knots, -0.04), marker='|', color='purple', s=100, label='Final (Learned) Knots')

plt.title('Fitting a B-Spline with Trainable Knots')
plt.xlabel('x')
plt.ylabel('Probability Density')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

In [ ]:
# ress.iloc[0]
final_control

In [ ]:
##################################
### run for ensemble_size = 5  ###
###       new noise method     ###
##################################

best_sols_matlab = loadmat("best_sols.mat")['best_sols']
best_sols_matlab = best_sols_matlab.reshape(best_sols_matlab.shape[0], 2, 3).swapaxes(1, 2)


xx = jnp.linspace(0, 4e-5, 300)

# num_rows = min(len(rilt_sols_list), len(final_sols))
# num_rows = len(final_clust_sols)
num_rows = len(quad_final_clust_sols)

plt.clf()
# fig, axs = plt.subplots(len(final_sols), 1, figsize=(16, 7*len(final_sols)), dpi=150)
fig, axs = plt.subplots(num_rows, 1, figsize=(16, 9*num_rows), dpi=150)

for i in range(num_rows):
    # observations = noisy_obs_list[i]
    observations = clean_obs_list[i]


    ####### use l-statistic or aic info? #######
    # n_srcs, _, _ = l_statistic2(final_full_sols[i], final_clust_sols[i], noisy_obs_list[i], q, t, sill_threshold=0.6, p_threshold=0.05)

    # n_srcs, _, _ = l_statistic(final_full_sols[i], final_clust_sols[i])
    # n_srcs, _, _ = l_statistic(quad_final_full_sols[i], quad_final_clust_sols[i])
    n_srcs = 2


    # ind = final_clust_sols[i]['aic_score'].argmin()
    # n_srcs = final_clust_sols[i].iloc[ind].name

    ############################################


    #### use centroids or use best solution? ###

    # amp_fin, D_fin, sig_fin = final_clust_sols[i].loc[n_srcs]['centers']
    amp_fin, D_fin, sig_fin = final_clust_sols[i].loc[n_srcs]['centers']
    if not isinstance(amp_fin, jnp.ndarray):
        amp_fin = jnp.array(amp_fin, ndmin=1)
    if not isinstance(D_fin, jnp.ndarray):
        D_fin = jnp.array(D_fin, ndmin=1)
    if not isinstance(sig_fin, jnp.ndarray):
        sig_fin = jnp.array(sig_fin, ndmin=1)
    beta_fin = 0.7

    amp_quad, D_quad, sig_quad = quad_final_clust_sols[i].loc[n_srcs]['centers']
    if not isinstance(amp_quad, jnp.ndarray):
        amp_quad = jnp.array(amp_quad, ndmin=1)
    if not isinstance(D_quad, jnp.ndarray):
        D_quad = jnp.array(D_quad, ndmin=1)
    if not isinstance(sig_quad, jnp.ndarray):
        sig_quad = jnp.array(sig_quad, ndmin=1)
    beta_fin = 0.7

    # amp_fin, D_fin, sig_fin, beta_fin = final_full_sols[i][final_full_sols[i]['num_sources'] == n_srcs].sort_values('fval').iloc[0]['sol']

    ############################################

    # pred_srcs = normal_distribution(xx, amp, D, sig_sol)
    true_amp, true_mu, true_std = valid_params[i]
    true_srcs = normal_distribution(xx, true_amp, true_mu, true_std)
    pred_srcs_final = normal_distribution(xx, amp_fin, D_fin, sig_fin)
    pred_srcs_quad = normal_distribution(xx, amp_quad, D_quad, sig_quad)

    clean_g2minus1 = g2_minus1_matrix(q, t, true_amp, true_mu, true_std, gen_beta)
    r = clean_g2minus1 - observations
    noise_level = 0.5 * jnp.sum(jnp.square(r))
    r = g2_minus1_matrix(q, t, amp_fin, D_fin, sig_fin, beta_fin) - observations
    fit_loss = 0.5*jnp.sum(jnp.square(r))

    # r = rilt_observation_matrix(q, t, possible_D, rilt_sols_list[i][0]) - observations
    # contin_loss = 0.5*jnp.sum(jnp.square(r))

    xx_ = xx * SCALING_CONST

    ax = axs[i]
    ax.plot(xx_, true_srcs, color='yellow', label='True distribution', linestyle=':')
    # ax.vlines(D, 0, amp*0.1, color='red', alpha=0.7, linestyle='-', label='phase 1 fit (dirac)')
    # ax.plot(xx, pred_srcs, color='teal', linestyle='-', label='phase 2 fit (width)')
    ax.plot(xx_, pred_srcs_final, color='green', linestyle='-', label='predicted distrubution (our model)')
    ax.plot(xx_, pred_srcs_quad, color='blue', linestyle='-.', label='predicted distrubution (quadrature)', alpha=0.5)

    # if i < len(best_sols_matlab):
    #     amp_matlab, D_matlab, sig_matlab = best_sols_matlab[i]
    #     pred_srcs_matlab = normal_distribution(xx, amp_matlab, D_matlab, sig_matlab)
    #     ax.plot(xx_, pred_srcs_matlab, color='turquoise', linestyle='-.', label='matlab code prediction')


    kl_our_model = jnp.sum(jax.scipy.special.rel_entr(true_srcs, pred_srcs_final))
    # rilt_eval_points = possible_D/SCALING_CONST
    # for j in range(len(rilt_sols_list[i])):
    #     rilt_sol = rilt_sols_list[i][j]
    #     rilt_sols_xx = jnp.interp(xx, rilt_eval_points, rilt_sol/10)
    #     if j == 0:
    #         label = 'CONTIN solutions (scaled by .1)'
    #         true_srcs_eval = normal_distribution(rilt_eval_points, true_amp, true_mu, true_std)
    #         kl_contin = jnp.sum(jax.scipy.special.rel_entr(true_srcs_eval, (rilt_sol + jnp.finfo(float).eps))/jnp.sum(rilt_sol))
    #     else:
    #         label = None
    #     ax.plot(xx_, rilt_sols_xx, alpha = 0.5, color='turquoise', linestyle='-.', label=label)



        # ax.plot(possible_D / SCALING_CONST, rilt_sol, linestyle='--', label=f'CONTIN solution {j+1}')
    # rilt_sol = rilt_sols_list[i][0] # top result
    # rilt_sols_xx = jnp.interp(xx, possible_D/SCALING_CONST, rilt_sol/10)
    # ax.plot(xx, rilt_sols_xx, linestyle='-.', label=f'best CONTIN solution (scaled by .1)')
    # ax.plot(possible_D / SCALING_CONST, rilt_sol, linestyle='-.', label=f'best CONTIN solution')

    ax.set_xlabel('Diffusion Coefficient (m^2/s)')

    ax.legend()
    # ax.set_title(f"bimodal mix {i} - noise level: {noise_level:.4e}, loss (our model): {fit_loss:.4e}, loss (best CONTIN): {contin_loss:.4e}\nKL divergences - (our model): {kl_our_model:.2e}, (best CONTIN fit): {kl_contin:.2e}")

    # ax.set_title(f"bimodal mix {i} - noise level: {noise_level:.4e}, loss (our model): {fit_loss:.4e}, loss (best CONTIN): {contin_loss:.4e}\nKL divergences - (our model): {kl_our_model:.2e}")
    ax.set_title(f"bimodal mix {i} - noise level: {noise_level:.4e}, loss (our model): {fit_loss:.4e}\nKL divergences - (our model): {kl_our_model:.2e}, num srcs: {n_srcs}")

plt.show()
